# ASR text embedding benchmark — Qwen3 Embedding 4B/8B

Notebook này tạo embedding ở **cấp ASR segment**. Một segment có thể bao phủ nhiều frame, vì vậy output giữ timestamp của segment và không tạo vector theo frame.

Artifact đầu ra gồm một file `.npy` và một file mapping `.jsonl` cho mỗi video, tách riêng theo model để hai dimension không bị trộn. Notebook này cũng đo model-load time, warmup time, encode latency và throughput; không gọi database, không sửa backend và không upload dữ liệu ra ngoài.

## 1. Install dependencies

Chạy cell này một lần trên Kaggle. Token Hugging Face nếu cần phải được lưu trong Kaggle Secrets với tên `HF_TOKEN`; không hard-code token vào notebook.

In [ ]:
%pip install -q --upgrade --no-cache-dir \
    "sentence-transformers>=3.0,<6" \
    "transformers>=4.51.0,<5" \
    "accelerate>=1.0.0,<2"

In [ ]:
import numpy
import scipy
import sklearn
import sentence_transformers

print(numpy.__version__)
print(scipy.__version__)
print(sklearn.__version__)
print(sentence_transformers.__version__)

## 2. Configuration

`INPUT_ROOT` có thể để `None` để tự tìm dataset đã attach trong `/kaggle/input`. Khi chạy local từ repository, notebook sẽ dùng `asr_output_corrected`. Hai model chạy tuần tự để giải phóng VRAM, output mặc định trên Kaggle là `/kaggle/working/asr_embedding_output_qwen3_v1/<model-key>/`.

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import torch
from tqdm.auto import tqdm


MODEL_SPECS = {
    "qwen3-embedding-4b": {
        "model_name": "Qwen/Qwen3-Embedding-4B",
        "embedding_dim": 2560,
        "batch_size": 8,
    },
    "qwen3-embedding-8b": {
        "model_name": "Qwen/Qwen3-Embedding-8B",
        "embedding_dim": 4096,
        "batch_size": 2,
    },
}
MODELS_TO_RUN = ["qwen3-embedding-4b", "qwen3-embedding-8b"]
MAX_SEQUENCE_LENGTH = 2048
ENCODE_CHUNK_SIZE = 4096
NORMALIZE_EMBEDDINGS = True
WARMUP_SEGMENTS = 32
USE_FLASH_ATTENTION_2 = False
# USE_DEVICE_MAP_AUTO = True
DEVICE = "cuda:0"
USE_DEVICE_MAP_AUTO = False

# Set an explicit path when the Kaggle dataset mount is known. None enables auto-discovery.
INPUT_ROOT = "/kaggle/input/datasets/nguyentranthienan/aic-2026-asr-corrected/asr_output_corrected_split"
OUTPUT_ROOT_BASE = None

# The notebook refuses to mix a new run with an existing model artifact directory.
# Choose another OUTPUT_ROOT_BASE for a new run instead of deleting old artifacts.
ALLOW_NONEMPTY_OUTPUT = False


def _default_input_root() -> Path:
    explicit = os.environ.get("ASR_INPUT_ROOT", "").strip()
    if explicit:
        return Path(explicit).expanduser()

    local_candidates = [
        Path.cwd() / "asr_output_corrected",
        Path("asr_output_corrected"),
    ]
    for candidate in local_candidates:
        if candidate.is_dir() and any(candidate.glob("*/asr_segments.jsonl")):
            return candidate

    kaggle_root = Path("/kaggle/input")
    if kaggle_root.is_dir():
        manifests = sorted(kaggle_root.rglob("asr_segments.jsonl"))
        if manifests:
            # A manifest normally lives at <input-root>/<batch>/asr_segments.jsonl.
            roots = sorted({manifest.parent.parent for manifest in manifests}, key=str)
            return roots[0]

    return Path("/kaggle/input/asr-output-corrected")


def _default_output_root() -> Path:
    explicit = os.environ.get("ASR_OUTPUT_ROOT", "").strip()
    if explicit:
        return Path(explicit).expanduser()
    if Path("/kaggle/working").is_dir():
        return Path("/kaggle/working/asr_embedding_output_qwen3_v1")
    return Path.cwd() / "asr_embedding_output_qwen3_v1"


def _read_hf_token() -> str:
    token = os.environ.get("HF_TOKEN", "").strip()
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient

        return (UserSecretsClient().get_secret("HF_TOKEN") or "").strip()
    except Exception:
        return ""


INPUT_ROOT = Path(INPUT_ROOT).expanduser() if INPUT_ROOT else _default_input_root()
OUTPUT_ROOT_BASE = Path(OUTPUT_ROOT_BASE).expanduser() if OUTPUT_ROOT_BASE else _default_output_root()
HF_TOKEN = _read_hf_token()
DEVICE = os.environ.get("ASR_EMBEDDING_DEVICE", "").strip()
DEVICE = DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("ASR_EMBEDDING_DEVICE requests CUDA, but CUDA is unavailable.")

print("INPUT_ROOT:      ", INPUT_ROOT)
print("OUTPUT_ROOT_BASE:", OUTPUT_ROOT_BASE)
print("MODELS_TO_RUN:   ", MODELS_TO_RUN)
print("DEVICE:          ", DEVICE)
print("BATCH_SIZES:     ", {key: MODEL_SPECS[key]["batch_size"] for key in MODELS_TO_RUN})
print("DEVICE_MAP_AUTO: ", USE_DEVICE_MAP_AUTO)
print("HF token loaded: ", bool(HF_TOKEN))
if DEVICE == "cpu":
    print("Warning: running on CPU. Enable a Kaggle GPU for the full dataset.")

for model_key in MODELS_TO_RUN:
    if model_key not in MODEL_SPECS:
        raise KeyError(f"Unknown model key: {model_key}")

## 3. Manifest discovery and conservative text cleanup

Text dùng để encode là `segment.text` (bản đã hiệu đính). `raw_text` và `normalized_text` vẫn được giữ nguyên trong mapping để audit. Cleanup chỉ xử lý whitespace và các n-gram lặp liên tiếp ít nhất 3 lần; không xóa segment khỏi artifact.

In [ ]:
_TOKEN_RE = re.compile(r"\S+")
_SAFE_ID_RE = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._-]*$")


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def optional_float(value: Any) -> float | None:
    if value is None or value == "":
        return None
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    return number if math.isfinite(number) else None


def required_float(value: Any, field_name: str) -> float:
    number = optional_float(value)
    if number is None:
        raise ValueError(f"{field_name} must be a finite number")
    return number


def token_key(token: str) -> str:
    return unicodedata.normalize("NFKC", token).casefold()


def collapse_repeated_ngrams(
    tokens: list[str], max_ngram_size: int = 8, min_repeats: int = 3
) -> tuple[list[str], bool, int, int]:
    """Collapse only adjacent repeated n-gram runs."""
    cleaned: list[str] = []
    repeat_detected = False
    repeat_runs = 0
    removed_tokens = 0
    index = 0

    while index < len(tokens):
        best: tuple[int, int, int] | None = None
        max_width = min(max_ngram_size, (len(tokens) - index) // min_repeats)

        for width in range(1, max_width + 1):
            phrase = [token_key(token) for token in tokens[index : index + width]]
            repeats = 1
            while index + (repeats + 1) * width <= len(tokens):
                candidate = [
                    token_key(token)
                    for token in tokens[index + repeats * width : index + (repeats + 1) * width]
                ]
                if candidate != phrase:
                    break
                repeats += 1

            if repeats >= min_repeats:
                covered = width * repeats
                # Prefer the run removing most tokens; for ties prefer the smallest phrase.
                candidate = (covered, width, repeats)
                if best is None or covered > best[0] or (covered == best[0] and width < best[1]):
                    best = candidate

        if best is None:
            cleaned.append(tokens[index])
            index += 1
            continue

        covered, width, repeats = best
        cleaned.extend(tokens[index : index + width])
        removed_tokens += covered - width
        repeat_runs += 1
        repeat_detected = True
        index += covered

    return cleaned, repeat_detected, repeat_runs, removed_tokens


def clean_embedding_text(value: Any) -> dict[str, Any]:
    original = unicodedata.normalize("NFC", str(value or ""))
    original = re.sub(r"\s+", " ", original).strip()
    tokens = _TOKEN_RE.findall(original)
    cleaned_tokens, repeated, repeat_runs, removed = collapse_repeated_ngrams(tokens)
    cleaned = " ".join(cleaned_tokens).strip()
    return {
        "embedding_text": cleaned,
        "raw_word_count": len(tokens),
        "clean_word_count": len(cleaned_tokens),
        "repeat_detected": repeated,
        "repeat_runs": repeat_runs,
        "repeat_tokens_removed": removed,
    }


def discover_manifest_files(input_root: Path) -> list[Path]:
    if not input_root.is_dir():
        raise FileNotFoundError(f"ASR input directory does not exist: {input_root}")
    manifests = sorted(input_root.rglob("asr_segments.jsonl"))
    if not manifests:
        raise FileNotFoundError(
            f"No asr_segments.jsonl found below {input_root}."
        )
    return manifests


def artifact_stem(video_id: str) -> str:
    if not _SAFE_ID_RE.fullmatch(video_id):
        raise ValueError(f"video_id is not safe for an artifact filename: {video_id!r}")
    return video_id


## 4. Load and flatten ASR segments

Mỗi top-level JSONL record là một video. Notebook giữ cả video không có segment bằng một mapping rỗng và vector matrix shape `(0, 1024)`. Các lỗi parse, duplicate ID, text rỗng hoặc timestamp không hợp lệ sẽ dừng run trước khi tải model.

In [ ]:
def load_asr_segments(input_root: Path) -> tuple[dict[str, dict[str, Any]], dict[str, Any]]:
    manifests = discover_manifest_files(input_root)
    videos: dict[str, dict[str, Any]] = {}
    seen_segment_ids: set[str] = set()
    parse_errors: list[str] = []
    validation_errors: list[str] = []

    for manifest_path in manifests:
        try:
            source_file = str(manifest_path.relative_to(input_root))
        except ValueError:
            source_file = manifest_path.name

        with manifest_path.open("r", encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue
                try:
                    video_record = json.loads(line)
                except json.JSONDecodeError as exc:
                    parse_errors.append(f"{source_file}:{line_number}: {exc}")
                    continue

                if not isinstance(video_record, dict):
                    validation_errors.append(f"{source_file}:{line_number}: record is not an object")
                    continue

                video_id = str(video_record.get("video_id") or "").strip()
                if not video_id:
                    validation_errors.append(f"{source_file}:{line_number}: missing video_id")
                    continue
                if video_id in videos:
                    validation_errors.append(f"duplicate video record: {video_id}")
                    continue

                video_info: dict[str, Any] = {
                    "video_id": video_id,
                    "batch_id": str(video_record.get("batch_id") or manifest_path.parent.name),
                    "dataset_code": video_record.get("dataset_code"),
                    "source_file": source_file,
                    "video_duration_seconds": optional_float((video_record.get("source") or {}).get("duration_seconds")),
                    "segments": [],
                }
                videos[video_id] = video_info

                segments = video_record.get("segments") or []
                if not isinstance(segments, list):
                    validation_errors.append(f"{source_file}:{line_number}: segments is not a list")
                    continue

                for segment_index, segment in enumerate(segments):
                    location = f"{source_file}:{line_number}:segment[{segment_index}]"
                    if not isinstance(segment, dict):
                        validation_errors.append(f"{location}: segment is not an object")
                        continue

                    segment_id = str(segment.get("segment_id") or "").strip()
                    if not segment_id:
                        validation_errors.append(f"{location}: missing segment_id")
                        continue
                    if segment_id in seen_segment_ids:
                        validation_errors.append(f"{location}: duplicate segment_id {segment_id}")
                        continue
                    seen_segment_ids.add(segment_id)

                    segment_video_id = str(segment.get("video_id") or video_id).strip()
                    if segment_video_id != video_id:
                        validation_errors.append(
                            f"{location}: segment video_id {segment_video_id!r} != {video_id!r}"
                        )
                        continue

                    text = str(segment.get("text") or "").strip()
                    if not text:
                        validation_errors.append(f"{location}: empty text")
                        continue
                    try:
                        start_seconds = required_float(segment.get("start_seconds"), "start_seconds")
                        end_seconds = required_float(segment.get("end_seconds"), "end_seconds")
                    except ValueError as exc:
                        validation_errors.append(f"{location}: {exc}")
                        continue
                    if start_seconds < 0 or end_seconds < start_seconds:
                        validation_errors.append(
                            f"{location}: invalid time range {start_seconds}..{end_seconds}"
                        )
                        continue

                    cleaned = clean_embedding_text(text)
                    if not cleaned["embedding_text"]:
                        validation_errors.append(f"{location}: cleanup produced empty text")
                        continue

                    record = {
                        "embedding_index_0": -1,
                        "segment_order_0": -1,
                        "segment_id": segment_id,
                        "video_id": video_id,
                        "batch_id": video_info["batch_id"],
                        "dataset_code": video_info["dataset_code"],
                        "start": start_seconds,
                        "end": end_seconds,
                        "start_seconds": start_seconds,
                        "end_seconds": end_seconds,
                        "timestamp_ms_start": segment.get("timestamp_ms_start"),
                        "timestamp_ms_end": segment.get("timestamp_ms_end"),
                        "text": text,
                        "embedding_text": cleaned["embedding_text"],
                        "normalized_text": str(segment.get("normalized_text") or ""),
                        "raw_text": str(segment.get("raw_text") or ""),
                        "status": str(segment.get("correction_status") or "unknown"),
                        "correction_status": str(segment.get("correction_status") or "unknown"),
                        "language": segment.get("language"),
                        "confidence": optional_float(segment.get("confidence")),
                        "avg_logprob": optional_float(segment.get("avg_logprob")),
                        "source_file": source_file,
                        "source_manifest_line": line_number,
                        "source_segment_index": segment_index,
                        "video_duration_seconds": video_info["video_duration_seconds"],
                        **cleaned,
                    }
                    video_info["segments"].append(record)

    if parse_errors or validation_errors:
        details = parse_errors[:10] + validation_errors[:20]
        raise RuntimeError(
            f"Input validation failed: {len(parse_errors)} parse errors, "
            f"{len(validation_errors)} validation errors.\n" + "\n".join(details)
        )

    for video_info in videos.values():
        video_info["segments"].sort(
            key=lambda item: (item["start_seconds"], item["end_seconds"], item["segment_id"])
        )
        for order, segment in enumerate(video_info["segments"]):
            segment["embedding_index_0"] = order
            segment["segment_order_0"] = order

    all_segments = [segment for video in videos.values() for segment in video["segments"]]
    summary = {
        "manifest_count": len(manifests),
        "video_count": len(videos),
        "segment_count": len(all_segments),
        "empty_video_count": sum(not video["segments"] for video in videos.values()),
        "empty_text_segment_count": 0,
        "repeat_detected_segment_count": sum(segment["repeat_detected"] for segment in all_segments),
        "status_counts": dict(Counter(segment["status"] for segment in all_segments)),
        "total_raw_word_count": sum(segment["raw_word_count"] for segment in all_segments),
        "total_clean_word_count": sum(segment["clean_word_count"] for segment in all_segments),
        "total_repeat_tokens_removed": sum(segment["repeat_tokens_removed"] for segment in all_segments),
    }
    return dict(sorted(videos.items())), summary


VIDEOS_BY_ID, LOAD_SUMMARY = load_asr_segments(INPUT_ROOT)
print(json.dumps(LOAD_SUMMARY, ensure_ascii=False, indent=2))
print("First videos:", list(VIDEOS_BY_ID)[:5])

### Preview flattened records

In [ ]:
preview = [
    segment
    for video_id in sorted(VIDEOS_BY_ID)
    for segment in VIDEOS_BY_ID[video_id]["segments"][:2]
][:5]
for item in preview:
    print(json.dumps({
        "segment_id": item["segment_id"],
        "video_id": item["video_id"],
        "start": item["start"],
        "end": item["end"],
        "text": item["text"],
        "embedding_text": item["embedding_text"],
        "repeat_detected": item["repeat_detected"],
    }, ensure_ascii=False))

## 5. Load Qwen3 models

Hai checkpoint được chạy tuần tự. Mỗi model có dimension, batch size và artifact directory riêng; `device_map=auto` có thể dùng nhiều GPU hoặc CPU offload khi cần.

In [ ]:
import gc
import time

import sentence_transformers
from sentence_transformers import SentenceTransformer


def synchronize_cuda() -> None:
    if not torch.cuda.is_available():
        return
    for index in range(torch.cuda.device_count()):
        torch.cuda.synchronize(index)


def reset_peak_memory_stats() -> None:
    if not torch.cuda.is_available():
        return
    for index in range(torch.cuda.device_count()):
        with torch.cuda.device(index):
            torch.cuda.reset_peak_memory_stats()


def peak_gpu_memory_gib() -> float | None:
    if not torch.cuda.is_available():
        return None
    synchronize_cuda()
    peak_bytes = max(
        torch.cuda.max_memory_allocated(index)
        for index in range(torch.cuda.device_count())
    )
    return float(peak_bytes / (1024**3))


def model_output_root(model_key: str) -> Path:
    return OUTPUT_ROOT_BASE / model_key


def load_qwen_model(model_key: str) -> tuple[SentenceTransformer, dict[str, Any]]:
    spec = MODEL_SPECS[model_key]
    use_device_map = USE_DEVICE_MAP_AUTO and DEVICE.startswith("cuda")
    model_kwargs: dict[str, Any] = {
        "torch_dtype": torch.float16 if DEVICE.startswith("cuda") else torch.float32,
    }
    if USE_FLASH_ATTENTION_2:
        model_kwargs["attn_implementation"] = "flash_attention_2"
    if use_device_map:
        model_kwargs["device_map"] = "auto"

    constructor_kwargs: dict[str, Any] = {
        "model_kwargs": model_kwargs,
        "tokenizer_kwargs": {"padding_side": "left"},
    }
    if not use_device_map:
        constructor_kwargs["device"] = DEVICE
    if HF_TOKEN:
        constructor_kwargs["token"] = HF_TOKEN

    model = SentenceTransformer(spec["model_name"], **constructor_kwargs)
    if hasattr(model, "max_seq_length"):
        model.max_seq_length = MAX_SEQUENCE_LENGTH
    model.eval()

    actual_dim = model.get_sentence_embedding_dimension()
    if actual_dim != spec["embedding_dim"]:
        raise ValueError(
            f"{model_key} returned dimension {actual_dim}; "
            f"expected {spec['embedding_dim']}"
        )
    return model, {
        "actual_dim": actual_dim,
        "dtype": str(model_kwargs["torch_dtype"]).replace("torch.", ""),
        "device_map_auto": use_device_map,
        "max_sequence_length": getattr(model, "max_seq_length", MAX_SEQUENCE_LENGTH),
    }


print("SentenceTransformers:", sentence_transformers.__version__)
print("Models:", {key: MODEL_SPECS[key] for key in MODELS_TO_RUN})
print("Max sequence length:", MAX_SEQUENCE_LENGTH)

## 6. Benchmark latency, encode segments and write artifacts

Mỗi model chạy một lần theo cùng thứ tự segment. Latency encode được đo bằng `perf_counter` với CUDA synchronization ở trước/sau mỗi chunk; model-load và warmup được báo cáo riêng, không cộng vào encode latency.

In [ ]:
def atomic_write_json(path: Path, payload: Any) -> None:
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
    temporary.replace(path)


def atomic_write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")))
            handle.write("\n")
    temporary.replace(path)


def atomic_save_npy(path: Path, array: np.ndarray) -> None:
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as handle:
        np.save(handle, array, allow_pickle=False)
    temporary.replace(path)


def benchmark_and_write(model_key: str) -> dict[str, Any]:
    spec = MODEL_SPECS[model_key]
    output_root = model_output_root(model_key)
    if output_root.exists() and any(output_root.iterdir()) and not ALLOW_NONEMPTY_OUTPUT:
        raise FileExistsError(
            f"Output directory is not empty: {output_root}. "
            "Set OUTPUT_ROOT_BASE to a new directory or explicitly enable ALLOW_NONEMPTY_OUTPUT."
        )
    (output_root / "embeddings").mkdir(parents=True, exist_ok=True)
    (output_root / "map-segments").mkdir(parents=True, exist_ok=True)

    ordered_video_ids = sorted(VIDEOS_BY_ID)
    ordered_segments = [
        segment
        for video_id in ordered_video_ids
        for segment in VIDEOS_BY_ID[video_id]["segments"]
    ]
    texts = [segment["embedding_text"] for segment in ordered_segments]
    model = None
    try:
        reset_peak_memory_stats()
        load_started = time.perf_counter()
        model, model_meta = load_qwen_model(model_key)
        synchronize_cuda()
        model_load_seconds = time.perf_counter() - load_started

        warmup_started = time.perf_counter()
        warmup_count = min(WARMUP_SEGMENTS, len(texts))
        if warmup_count:
            model.encode(
                texts[:warmup_count],
                batch_size=spec["batch_size"],
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=NORMALIZE_EMBEDDINGS,
            )
            synchronize_cuda()
        warmup_seconds = time.perf_counter() - warmup_started

        vector_chunks: list[np.ndarray] = []
        chunk_latencies_ms: list[float] = []
        encode_started = time.perf_counter()
        for offset in tqdm(
            range(0, len(texts), ENCODE_CHUNK_SIZE),
            desc=f"Encoding {model_key}",
        ):
            chunk_texts = texts[offset : offset + ENCODE_CHUNK_SIZE]
            synchronize_cuda()
            chunk_started = time.perf_counter()
            chunk = model.encode(
                chunk_texts,
                batch_size=spec["batch_size"],
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=NORMALIZE_EMBEDDINGS,
            )
            synchronize_cuda()
            chunk_latencies_ms.append((time.perf_counter() - chunk_started) * 1000.0)
            chunk = np.asarray(chunk, dtype=np.float32)
            if chunk.ndim == 1:
                chunk = chunk.reshape(1, -1)
            if chunk.shape != (len(chunk_texts), spec["embedding_dim"]):
                raise ValueError(
                    f"Unexpected {model_key} chunk shape {chunk.shape}; expected "
                    f"({len(chunk_texts)}, {spec['embedding_dim']})"
                )
            if not np.isfinite(chunk).all():
                raise ValueError(f"Non-finite value found in {model_key} chunk at {offset}")
            vector_chunks.append(chunk)
        encode_seconds = time.perf_counter() - encode_started

        if vector_chunks:
            all_vectors = np.concatenate(vector_chunks, axis=0).astype(np.float32, copy=False)
        else:
            all_vectors = np.empty((0, spec["embedding_dim"]), dtype=np.float32)
        if all_vectors.shape != (len(ordered_segments), spec["embedding_dim"]):
            raise ValueError(f"Global {model_key} vector shape mismatch: {all_vectors.shape}")

        cursor = 0
        for video_id in tqdm(ordered_video_ids, desc=f"Writing {model_key}"):
            segments = VIDEOS_BY_ID[video_id]["segments"]
            count = len(segments)
            video_vectors = all_vectors[cursor : cursor + count]
            cursor += count
            stem = artifact_stem(video_id)
            atomic_save_npy(output_root / "embeddings" / f"{stem}.npy", video_vectors)
            atomic_write_jsonl(output_root / "map-segments" / f"{stem}.jsonl", segments)
        assert cursor == len(ordered_segments)

        synchronize_cuda()
        encode_seconds = float(encode_seconds)
        result = {
            "model_key": model_key,
            "model_name": spec["model_name"],
            "embedding_dim": spec["embedding_dim"],
            "device": DEVICE,
            "dtype": model_meta["dtype"],
            "device_map_auto": model_meta["device_map_auto"],
            "max_sequence_length": model_meta["max_sequence_length"],
            "batch_size": spec["batch_size"],
            "encode_chunk_size": ENCODE_CHUNK_SIZE,
            "num_videos": len(ordered_video_ids),
            "num_segments": len(ordered_segments),
            "model_load_seconds": float(model_load_seconds),
            "warmup_seconds": float(warmup_seconds),
            "warmup_segments": warmup_count,
            "encode_seconds": encode_seconds,
            "latency_ms_per_segment": (encode_seconds * 1000.0 / len(ordered_segments)) if ordered_segments else 0.0,
            "throughput_segments_per_second": (len(ordered_segments) / encode_seconds) if encode_seconds > 0 else 0.0,
            "chunk_count": len(chunk_latencies_ms),
            "chunk_latency_ms_p50": float(np.percentile(chunk_latencies_ms, 50)) if chunk_latencies_ms else 0.0,
            "chunk_latency_ms_p95": float(np.percentile(chunk_latencies_ms, 95)) if chunk_latencies_ms else 0.0,
            "chunk_latency_ms_p99": float(np.percentile(chunk_latencies_ms, 99)) if chunk_latencies_ms else 0.0,
            "chunk_latency_ms": chunk_latencies_ms,
            "latency_definition": "synchronized wall-clock encode time, excluding model load and warmup",
            "peak_gpu_memory_gib": peak_gpu_memory_gib(),
            "output_root": str(output_root),
        }
        return result
    finally:
        if model is not None:
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


BENCHMARK_RESULTS: dict[str, dict[str, Any]] = {}
for model_key in MODELS_TO_RUN:
    print("\n" + "=" * 80)
    print("RUNNING:", model_key, MODEL_SPECS[model_key]["model_name"])
    BENCHMARK_RESULTS[model_key] = benchmark_and_write(model_key)
    print(json.dumps(BENCHMARK_RESULTS[model_key], ensure_ascii=False, indent=2))

## 7. Validate the generated artifact

Validation đọc lại output của từng model từ disk để bảo đảm `.npy` và `.jsonl` khớp theo số dòng, thứ tự segment, dimension, dtype, finite values và norm.

In [ ]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise AssertionError(f"Invalid JSON at {path}:{line_number}: {exc}") from exc
            if not isinstance(row, dict):
                raise AssertionError(f"Mapping row is not an object: {path}:{line_number}")
            rows.append(row)
    return rows


def validate_outputs(model_key: str) -> dict[str, Any]:
    spec = MODEL_SPECS[model_key]
    output_root = model_output_root(model_key)
    embedding_dir = output_root / "embeddings"
    mapping_dir = output_root / "map-segments"
    expected_stems = {artifact_stem(video_id) for video_id in VIDEOS_BY_ID}
    actual_embedding_stems = {path.stem for path in embedding_dir.glob("*.npy")}
    actual_mapping_stems = {path.stem for path in mapping_dir.glob("*.jsonl")}

    assert actual_embedding_stems == expected_stems, (
        model_key, "Embedding file set mismatch", expected_stems - actual_embedding_stems,
        actual_embedding_stems - expected_stems
    )
    assert actual_mapping_stems == expected_stems, (
        model_key, "Mapping file set mismatch", expected_stems - actual_mapping_stems,
        actual_mapping_stems - expected_stems
    )

    total_vectors = 0
    total_mapping_rows = 0
    max_norm_error = 0.0

    for video_id in sorted(VIDEOS_BY_ID):
        stem = artifact_stem(video_id)
        matrix = np.load(embedding_dir / f"{stem}.npy", mmap_mode="r", allow_pickle=False)
        mapping = read_jsonl(mapping_dir / f"{stem}.jsonl")
        expected_segments = VIDEOS_BY_ID[video_id]["segments"]

        assert matrix.ndim == 2, (model_key, video_id, matrix.shape)
        assert matrix.dtype == np.float32, (model_key, video_id, matrix.dtype)
        assert matrix.shape == (len(expected_segments), spec["embedding_dim"]), (
            model_key, video_id, matrix.shape, len(expected_segments)
        )
        assert len(mapping) == matrix.shape[0], (model_key, video_id, len(mapping), matrix.shape[0])
        assert np.isfinite(matrix).all(), f"Non-finite vector in {model_key}/{video_id}"

        expected_ids = [segment["segment_id"] for segment in expected_segments]
        actual_ids = [row.get("segment_id") for row in mapping]
        assert actual_ids == expected_ids, f"Segment order mismatch in {model_key}/{video_id}"
        assert len(actual_ids) == len(set(actual_ids)), f"Duplicate mapping IDs in {model_key}/{video_id}"

        for index, row in enumerate(mapping):
            assert row.get("embedding_index_0") == index, (model_key, video_id, index, row)
            assert row.get("segment_order_0") == index, (model_key, video_id, index, row)
            assert row.get("video_id") == video_id, (model_key, video_id, row)
            assert row.get("text") is not None
            assert row.get("embedding_text")
            assert row.get("source_file")

        if matrix.shape[0] and NORMALIZE_EMBEDDINGS:
            norms = np.linalg.norm(np.asarray(matrix), axis=1)
            assert np.all(norms > 1e-8), f"Zero vector in {model_key}/{video_id}"
            norm_error = float(np.max(np.abs(norms - 1.0)))
            max_norm_error = max(max_norm_error, norm_error)
            assert norm_error <= 1e-3, (model_key, video_id, norm_error)

        total_vectors += matrix.shape[0]
        total_mapping_rows += len(mapping)

    temporary_files = list(output_root.rglob("*.tmp"))
    assert not temporary_files, f"Temporary files remain in {output_root}: {temporary_files[:5]}"
    assert total_vectors == LOAD_SUMMARY["segment_count"]
    assert total_mapping_rows == total_vectors

    return {
        "model_key": model_key,
        "model_name": spec["model_name"],
        "video_count": len(VIDEOS_BY_ID),
        "vector_count": total_vectors,
        "mapping_row_count": total_mapping_rows,
        "embedding_dim": spec["embedding_dim"],
        "max_l2_norm_error": max_norm_error,
    }


VALIDATION_REPORTS: dict[str, dict[str, Any]] = {}
for model_key in MODELS_TO_RUN:
    VALIDATION_REPORTS[model_key] = validate_outputs(model_key)
    print(json.dumps(VALIDATION_REPORTS[model_key], ensure_ascii=False, indent=2))

## 8. Write per-model metadata, latency benchmark and completion markers

Mỗi model có `model_info.json`, `summary.json` và `_SUCCESS` riêng. File `latency_benchmark.json` ở root tổng hợp hai model; `_SUCCESS` ở root chỉ được tạo sau khi cả hai model đã validate thành công.

In [ ]:
artifact_schema_version = "aic.asr_text_embedding_benchmark.v1"
created_at = utc_now_iso()
run_id = "asr_text_embedding_qwen3_v1_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
all_segments = [segment for video in VIDEOS_BY_ID.values() for segment in video["segments"]]
status_counts = dict(Counter(segment["status"] for segment in all_segments))
batch_counts: dict[str, dict[str, int]] = defaultdict(lambda: {"videos": 0, "segments": 0})
for video in VIDEOS_BY_ID.values():
    batch = str(video["batch_id"])
    batch_counts[batch]["videos"] += 1
    batch_counts[batch]["segments"] += len(video["segments"])

for model_key in MODELS_TO_RUN:
    spec = MODEL_SPECS[model_key]
    output_root = model_output_root(model_key)
    latency = BENCHMARK_RESULTS[model_key]
    validation = VALIDATION_REPORTS[model_key]
    model_info = {
        "artifact_schema_version": artifact_schema_version,
        "run_id": run_id,
        "created_at": created_at,
        "model_key": model_key,
        "model_name": spec["model_name"],
        "model_version": spec["model_name"],
        "sentence_transformers_version": sentence_transformers.__version__,
        "embedding_dim": spec["embedding_dim"],
        "dtype": latency["dtype"],
        "normalize_embeddings": NORMALIZE_EMBEDDINGS,
        "similarity_metric": "cosine",
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "source_schema_version": "aic.asr_artifact.v1",
        "source_text_field": "text",
        "encoded_text_field": "embedding_text",
        "output_layout": {
            "embeddings": "embeddings/<video_id>.npy",
            "mapping": "map-segments/<video_id>.jsonl",
            "row_key": "embedding_index_0",
        },
        "latency": latency,
    }
    summary = {
        "status": "success",
        "artifact_schema_version": artifact_schema_version,
        "run_id": run_id,
        "created_at": created_at,
        "model_key": model_key,
        "model_name": spec["model_name"],
        "input_root": str(INPUT_ROOT),
        "output_root": str(output_root),
        "manifest_count": LOAD_SUMMARY["manifest_count"],
        "video_count": LOAD_SUMMARY["video_count"],
        "empty_video_count": LOAD_SUMMARY["empty_video_count"],
        "segment_count": LOAD_SUMMARY["segment_count"],
        "vector_count": validation["vector_count"],
        "mapping_row_count": validation["mapping_row_count"],
        "embedding_dim": validation["embedding_dim"],
        "empty_text_segment_count": LOAD_SUMMARY["empty_text_segment_count"],
        "repeat_detected_segment_count": LOAD_SUMMARY["repeat_detected_segment_count"],
        "total_raw_word_count": LOAD_SUMMARY["total_raw_word_count"],
        "total_clean_word_count": LOAD_SUMMARY["total_clean_word_count"],
        "total_repeat_tokens_removed": LOAD_SUMMARY["total_repeat_tokens_removed"],
        "status_counts": status_counts,
        "batch_counts": dict(sorted(batch_counts.items())),
        "validation": validation,
        "latency": latency,
    }
    atomic_write_json(output_root / "model_info.json", model_info)
    atomic_write_json(output_root / "summary.json", summary)
    atomic_write_json(
        output_root / "_SUCCESS",
        {
            "status": "success",
            "artifact_schema_version": artifact_schema_version,
            "run_id": run_id,
            "model_key": model_key,
            "vector_count": validation["vector_count"],
        },
    )

benchmark_summary = {
    "status": "success",
    "artifact_schema_version": artifact_schema_version,
    "run_id": run_id,
    "created_at": created_at,
    "input_root": str(INPUT_ROOT),
    "output_root": str(OUTPUT_ROOT_BASE),
    "models": {model_key: BENCHMARK_RESULTS[model_key] for model_key in MODELS_TO_RUN},
    "validation": VALIDATION_REPORTS,
    "comparison_fields": [
        "model_load_seconds",
        "encode_seconds",
        "latency_ms_per_segment",
        "throughput_segments_per_second",
        "chunk_latency_ms_p50",
        "chunk_latency_ms_p95",
        "chunk_latency_ms_p99",
        "peak_gpu_memory_gib",
    ],
}
atomic_write_json(OUTPUT_ROOT_BASE / "latency_benchmark.json", benchmark_summary)
atomic_write_json(
    OUTPUT_ROOT_BASE / "_SUCCESS",
    {
        "status": "success",
        "artifact_schema_version": artifact_schema_version,
        "run_id": run_id,
        "models": MODELS_TO_RUN,
    },
)

print("Benchmark completed:", OUTPUT_ROOT_BASE)
print(json.dumps(benchmark_summary, ensure_ascii=False, indent=2))

## 9. Inspect the benchmark output

In [ ]:
for model_key in MODELS_TO_RUN:
    output_root = model_output_root(model_key)
    print("\n" + "=" * 80)
    print(model_key, "->", output_root)
    for path in sorted(output_root.rglob("*")):
        if path.is_file():
            print(path.relative_to(output_root), f"({path.stat().st_size:,} bytes)")

    first_video = sorted(VIDEOS_BY_ID)[0] if VIDEOS_BY_ID else None
    if first_video:
        first_map = output_root / "map-segments" / f"{artifact_stem(first_video)}.jsonl"
        first_rows = read_jsonl(first_map)[:2]
        print("Sample mapping rows:")
        print(json.dumps(first_rows, ensure_ascii=False, indent=2))

print("\nLatency comparison:")
for model_key in MODELS_TO_RUN:
    result = BENCHMARK_RESULTS[model_key]
    print(
        f"{model_key}: encode={result['encode_seconds']:.3f}s, "
        f"{result['latency_ms_per_segment']:.3f} ms/segment, "
        f"{result['throughput_segments_per_second']:.2f} segments/s"
    )